# Remove First Week from `grainger_chat_traces_gold`

The week of **2025-11-10** has 100% `final_state_extractable = false` (675,817 records),
indicating the extraction pipeline was not yet operational. This notebook removes that week
from the Delta table.

**Table:** `ctlg_insight_prod_shared_gold.schm_cscdigitalassistant.grainger_chat_traces_gold`

In [ ]:
TABLE = "ctlg_insight_prod_shared_gold.schm_cscdigitalassistant.grainger_chat_traces_gold"

## 1. Diagnostic — Identify the earliest week and confirm it is 100% not extractable

In [ ]:
diagnostic_df = spark.sql(f"""
    SELECT
        `final_state_extractable`,
        date_trunc('week', end_time) AS week,
        COUNT(*) AS row_count
    FROM {TABLE}
    WHERE date_trunc('week', end_time) = (
        SELECT MIN(date_trunc('week', end_time)) FROM {TABLE}
    )
    GROUP BY `final_state_extractable`, date_trunc('week', end_time)
    ORDER BY `final_state_extractable`
""")

display(diagnostic_df)

## 2. Delete all rows from the earliest week

In [ ]:
min_week_df = spark.sql(f"""
    SELECT MIN(date_trunc('week', end_time)) AS min_week FROM {TABLE}
""")
display(min_week_df)

deleted = spark.sql(f"""
    DELETE FROM {TABLE}
    WHERE date_trunc('week', end_time) = (
        SELECT MIN(date_trunc('week', end_time)) FROM {TABLE}
    )
""")

display(deleted)

## 3. Verify — Confirm the earliest week is now 2025-11-17

In [ ]:
verification_df = spark.sql(f"""
    WITH weekly_counts AS (
        SELECT
            `final_state_extractable`,
            date_trunc('week', end_time) AS week,
            COUNT(*) AS count
        FROM {TABLE}
        GROUP BY `final_state_extractable`, date_trunc('week', end_time)
    ),
    total_weekly_counts AS (
        SELECT week, SUM(count) AS total_count
        FROM weekly_counts
        GROUP BY week
    )
    SELECT
        wc.`final_state_extractable`,
        wc.week,
        wc.count,
        (wc.count * 100.0) / twc.total_count AS percentage
    FROM weekly_counts wc
    JOIN total_weekly_counts twc ON wc.week = twc.week
    ORDER BY wc.week, wc.`final_state_extractable`
""")

display(verification_df)

## 4. Delta Time Travel — Restore if needed

Delta Lake keeps a transaction log. If you need to undo the delete, use one of the cells below.

First, check the table history to find the version **before** the delete:

In [ ]:
history_df = spark.sql(f"DESCRIBE HISTORY {TABLE} LIMIT 5")
display(history_df)

Uncomment and run **one** of the restore options below, replacing the placeholder with the correct version or timestamp from the history above.

In [ ]:
# Option A: Restore by version number (find the version before the DELETE in the history above)
# spark.sql(f"RESTORE TABLE {TABLE} TO VERSION AS OF <version_number>")

# Option B: Restore by timestamp (use a timestamp before you ran the DELETE)
# spark.sql(f"RESTORE TABLE {TABLE} TO TIMESTAMP AS OF '2025-03-25T00:00:00'")